In [1]:
!pip install scipy
import gymnasium as gym
import torch
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import cv2  # OpenCV for image preprocessing
from collections import deque
import ale_py
print(ale_py.__version__)
!pip install seaborn
import time
import wrapt
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
from IPython.display import clear_output
!pip install torch torchvision torchaudio
import torch
import torch.nn as nn
import torch.optim as optim
import random
import torch.nn.functional as F
from collections import deque
# Supress numpy scientific notation
np.set_printoptions(suppress=True)

0.10.2


In [2]:
import os
print(os.getcwd())  # Check the current working directory
print(os.listdir())  # List files in that directory
print("Current working directory:", os.getcwd())  # Verify change

c:\Users\David\OneDrive\Desktop\DSBA 6010 DRL
['02a-RL-Review-MDP.pdf', '1_point_0.png', '1_point_1.png', '1_point_3.png', '3.png', '3_1.png', '3_2.png', '4_point_5.png', '4_point_5_1.png', '4_point_5_2.png', '6010DRL Textbook.pdf', 'A1demo_Discrete_Marble.ipynb', 'another_point_one_0.png', 'another_point_one_1.png', 'another_point_one_2.png', 'assign1.html', 'assign1.ipynb', 'assign1_files', 'assign2.ipynb', 'assignment 1 analysis notes.txt', 'assignment 1 analysis.docx', 'assignment 1 analysis.zip', 'assignment1_dbayha.ipynb', 'attempt 1.txt', 'attempt 3.txt', 'best_model.pth', 'best_modelparam1.pth', 'best_modelparam3.pth', 'best_modelparam4.pth', 'best_modelparam5.pth', 'Citations_assignment_1.docx', 'DQN PONG CODE BREAKDOWN.docx', 'DQN update_1.ipynb', 'DQN update_2.ipynb', 'DQN update_3.ipynb', 'DQN update_4.ipynb', 'DQN_update_5.ipynb', 'DQN_update_6.ipynb', 'error.png', 'first positive reward.png', 'first_0.png', 'first_1.png', 'first_2.png', 'gamma_point_five_0.png', 'gamma_po

In [3]:
import os
os.chdir(r"C:\Users\David\Downloads")  # Change directory to Downloads
print("Current working directory:", os.getcwd())  # Verify change


Current working directory: C:\Users\David\Downloads


In [ ]:
#Creating DQN Agent
class DQN(nn.Module):
    """This is the constructor of the class that takes the dimensions of the game and
    the number of possible actions the agent can take as input."""
    def __init__(self, input_shape, n_actions = 6):
        super(DQN, self).__init__()
        """These are the convolutional layers of our CNN. They are responsible for processing 
        the visual informationm from the game. """
        self.conv = nn.Sequential(
            nn.Conv2d(input_shape[0], 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU()
        )
        conv_out_size = self._get_conv_out(input_shape)
        #Here we define the fully connected layers, which are known as dense layers.
        self.fc = nn.Sequential(
            nn.Linear(conv_out_size, 512),
            nn.ReLU(),
            nn.Linear(512, n_actions)
        )
    """Here we define a helper function that calculates the output size of the 
    convolutional layers. The size is needed to correctly define the input size of the first 
    fully connected layer."""
    def _get_conv_out(self, shape):
        dummy_input = torch.zeros(1, *shape).to(next(self.parameters()).device)
        o = self.conv(torch.zeros(1, *shape))
        return int(np.prod(o.size()))
    """This is the most important function for our network because it defines how the input
    data goes through the network. By taking the game screen as input it returns Q-values for 
    each possible action."""
    def forward(self, x):
        x = x.float()
        conv_out = self.conv(x)
        conv_out = conv_out.view(conv_out.size(0), -1)
        return self.fc(conv_out)


In [5]:
def evaluate(env, policy_net, device, n_episodes=10, render=True, verbose=True):
    """
    Testing for the trained DQN agent to evaluate quality of hyper parameters.
    
    Args:
        env: The Pong environment
        policy_net: Trained DQN model
        device: Device to run the model on
        n_episodes: Number of evaluation episodes
        render: Whether to render the environment
        verbose: Whether to print detailed results
    
    Returns:
        mean_reward: Average reward across evaluation episodes
        std_reward: Standard deviation of rewards
    """
    policy_net.eval()  # Set to evaluation mode
    eval_rewards = []
    
    for episode in range(n_episodes):
        state, _ = env.reset()
        done = False
        total_reward = 0
        
        while not done:
            # Get action from policy network
            with torch.no_grad():
                state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
                action = policy_net(state_tensor).max(1)[1].item()
            
            # Take step in environment
            state, reward, done, truncated, _ = env.step(action)
            total_reward += reward
            done = done or truncated
            
            if render:
                env.render()
        
        eval_rewards.append(total_reward)
        if verbose:
            print(f"Evaluation Episode {episode + 1}/{n_episodes} | Reward: {total_reward:.2f}")
    
    mean_reward = np.mean(eval_rewards)
    std_reward = np.std(eval_rewards)
    
    if verbose:
        print("\n=== Evaluation Results ===")
        print(f"Average Reward: {mean_reward:.2f} ± {std_reward:.2f}")
        print(f"Min Reward: {np.min(eval_rewards):.2f}")
        print(f"Max Reward: {np.max(eval_rewards):.2f}")
    
    return mean_reward, std_reward

In [1]:
import gymnasium as gym
import torch
import numpy as np
import cv2

#Create and define the Preprocessing Function
class PreprocessEnv(gym.Wrapper):
    def __init__(self, env, skip=4):
        super(PreprocessEnv, self).__init__(env)
        self._skip = skip
    """ The step method executes the given action for skip frames and accumulates rewards. We
    do this so that the number of frames the agent tneeds to process is reduced and the agent can
    learn faster."""
    def step(self, action):
        total_reward = 0.0
        done = False
        truncated = False
        for i in range(self._skip):
            obs, reward, done, truncated, info = self.env.step(action)
            total_reward += reward
            if done:
                break
        
        # Process observation
        obs = self._preprocess_observation(obs)
        return obs, total_reward, done, truncated, info

    def reset(self):
        obs, info = self.env.reset()
        obs = self._preprocess_observation(obs)  
        return obs, info
    """ We are defining the function that converts the color image observation to grayscale 
    by using cv2.cvtColor. This is done to reduce the complexity of the input data."""
    def _preprocess_observation(self, obs):
        # Convert to grayscale
        obs = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)
        # Resize the image to 84x84 pixels to reduce input size
        obs = cv2.resize(obs, (84, 84), interpolation=cv2.INTER_AREA)
        # Normalize pixel values by dividing by 255 to fit in the range [0, 1] for CNN 
        obs = obs / 255.0
        return obs
"""The FrameStack class enhances the agent's comprehension of the game ennviornnment by stacking
consecutive frames together.  This is essential to do becuase the speed and direction of the ball
can't be determined from a single frame. This allows the agent to correctly perceive motion and
changes over time."""
class FrameStack(gym.Wrapper):
    def __init__(self, env, k):
        super(FrameStack, self).__init__(env)
        self.k = k #The number of frames to stack
        self.frames = deque([], maxlen=k) #Double-ended queue with max length of k
        shp = env.observation_space.shape
        """Now we adjust the format of the observations that the agent will receive to
        account for the stacked frames """
        self.observation_space = gym.spaces.Box(
            low=0,
            high=1,
            shape=(k, shp[0], shp[1]),
            dtype=np.float32
        )
    """Define reset function to initialize the stack with the starting state. Once reset the environment
    receives the initial observastion ands appends it to the self.frames deque k times."""
    def reset(self):
        obs, info = self.env.reset()
        for _ in range(self.k):
            self.frames.append(obs)
        return self._get_obs(), info
    # Steps the environment and the new observation is added to self.frames
    def step(self, action):
        obs, reward, done, truncated, info = self.env.step(action)
        self.frames.append(obs)
        return self._get_obs(), reward, done, truncated, info
    # This is a helper function that converts the deque of frames into a NumPy array
    def _get_obs(self):
        return np.array(self.frames)

Initializing...
Applying preprocessing...
Evaluation Episode 1/50 | Reward: 7.00
Evaluation Episode 2/50 | Reward: -6.00
Evaluation Episode 3/50 | Reward: -7.00
Evaluation Episode 4/50 | Reward: 14.00
Evaluation Episode 5/50 | Reward: -5.00
Evaluation Episode 6/50 | Reward: -4.00
Evaluation Episode 7/50 | Reward: 10.00
Evaluation Episode 8/50 | Reward: 5.00
Evaluation Episode 9/50 | Reward: 12.00
Evaluation Episode 10/50 | Reward: -8.00
Evaluation Episode 11/50 | Reward: 7.00
Evaluation Episode 12/50 | Reward: 12.00
Evaluation Episode 13/50 | Reward: 10.00
Evaluation Episode 14/50 | Reward: -2.00
Evaluation Episode 15/50 | Reward: 3.00
Evaluation Episode 16/50 | Reward: 3.00
Evaluation Episode 17/50 | Reward: 10.00
Evaluation Episode 18/50 | Reward: 11.00
Evaluation Episode 19/50 | Reward: 11.00
Evaluation Episode 20/50 | Reward: 3.00
Evaluation Episode 21/50 | Reward: 2.00
Evaluation Episode 22/50 | Reward: 7.00
Evaluation Episode 23/50 | Reward: 12.00
Evaluation Episode 24/50 | Rewar

KeyboardInterrupt: 

: 

In [ ]:
# Set up device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize environment
print("Initializing...")
env = gym.make("ALE/Pong-v5", render_mode="human")
print("Applying preprocessing...")
env = PreprocessEnv(env)

env = FrameStack(env, 4)

# Only need the policy network
policy_net = DQN(input_shape=(4, 84, 84), n_actions=6).to(device)
policy_net.load_state_dict(torch.load('best_model2.pth'))
policy_net.eval()

# Run evaluation
mean_reward, std_reward = evaluate(
    env=env,
    policy_net=policy_net,
    device=device,
    n_episodes=50,
    render=True,
    verbose=True
)

env.close()

Initializing...
Applying preprocessing...
Evaluation Episode 1/50 | Reward: 6.00
Evaluation Episode 2/50 | Reward: 10.00
Evaluation Episode 3/50 | Reward: 12.00
Evaluation Episode 4/50 | Reward: 4.00
Evaluation Episode 5/50 | Reward: 4.00
Evaluation Episode 6/50 | Reward: 8.00
Evaluation Episode 7/50 | Reward: 3.00
Evaluation Episode 8/50 | Reward: -2.00
Evaluation Episode 9/50 | Reward: 9.00
Evaluation Episode 10/50 | Reward: 1.00
Evaluation Episode 11/50 | Reward: 3.00
Evaluation Episode 12/50 | Reward: 2.00
Evaluation Episode 13/50 | Reward: 6.00
Evaluation Episode 14/50 | Reward: 4.00
Evaluation Episode 15/50 | Reward: 11.00
Evaluation Episode 16/50 | Reward: 10.00
Evaluation Episode 17/50 | Reward: 15.00
Evaluation Episode 18/50 | Reward: 11.00
Evaluation Episode 19/50 | Reward: 9.00
Evaluation Episode 20/50 | Reward: 6.00
Evaluation Episode 21/50 | Reward: 2.00
Evaluation Episode 22/50 | Reward: -4.00
Evaluation Episode 23/50 | Reward: 10.00
Evaluation Episode 24/50 | Reward: 9.0